In [42]:
import fitz 
import json
from pathlib import Path
import re


In [44]:
PDF_PATH = Path("./boletim/6/BHA_PT_1.pdf")

In [ ]:
template = json.loads(TEMPLATE_JSON_PATH.read_text(encoding="utf-8"))

In [72]:
def clean_text(s: str) -> str:
    s = s.replace("\u00ad", "")  # soft hyphen
    s = s.replace("\ufb01", "fi").replace("\ufb02", "fl")  # ligatures
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def extract_pages_text(pdf_path: Path) -> list[dict]:
    doc = fitz.open(pdf_path)
    pages = []
    for i in range(doc.page_count):
        txt = doc.load_page(i).get_text("text")
        pages.append({"page": i + 1, "text": clean_text(txt)})
    doc.close()
    return pages

def parse_front_matter(page1_text: str) -> dict:
    # Ex.: "DOI:10.61818/02910606 ISSN: 2965-0291" + "Volume 6, Número 06 Manaus, 11 de fevereiro de 2026"
    out = {}
    doi = re.search(r"\bDOI\s*:\s*([0-9.]+\/[0-9]+)\b", page1_text, flags=re.I)
    issn = re.search(r"\bISSN\s*:\s*([0-9]{4}-[0-9]{4})\b", page1_text, flags=re.I)
    vol_num = re.search(r"\bVolume\s*([0-9]+)\s*,\s*N[uú]mero\s*([0-9]+)\b", page1_text, flags=re.I)
    date_line = re.search(r"\bManaus\s*,\s*(.+)$", page1_text, flags=re.I | re.M)

    if doi:
        out["doi"] = doi.group(1).strip()
    if issn:
        out["issn"] = issn.group(1).strip()
    if vol_num:
        out["volume"] = vol_num.group(1).zfill(1)
        out["number"] = vol_num.group(2).zfill(2)
    if date_line:
        out["date"] = date_line.group(1).strip()

    # Título principal
    # (pegamos linhas “Boletim ... Bacia Amazônica”)
    title_lines = []
    for line in page1_text.splitlines():
        line = line.strip()
        if not line:
            continue
        if "DOI:" in line.upper() or "ISSN:" in line.upper():
            continue
        if re.search(r"\bVolume\b", line, flags=re.I):
            continue
        # linhas de título geralmente grandes e sem pontuação
        title_lines.append(line)
    # tenta montar o título até antes de "Volume"
    if title_lines:
        out["title"] = " ".join(title_lines).strip()

    return out

def parse_current_conditions(text_page4: str) -> dict:
    # A seção começa com "Condições atuais" e em seguida vem um parágrafo descritivo.
    # Vamos capturar do título até antes do bloco numerado das bacias (1 Abacaxis ...).
    m = re.search(r"Condi[cç][oõ]es\s+atuais\s*(.+?)(?=\n1\s+Abacaxis|\Z)", text_page4, flags=re.I | re.S)
    desc = clean_text(m.group(1)) if m else clean_text(text_page4)
    return {
        "text": desc,
        # você pode completar com caminhos/nomes de imagens se também extrair figuras:
        "map_current_conditions": None,
        "table_current_conditions": None,
    }
    
def get_section_text(pages_text: list[dict], start_page: int, end_page: int) -> str:
    parts = []
    for p in pages_text:
        if start_page <= p["page"] <= end_page:
            parts.append(p["text"])
    return clean_text("\n\n".join(parts))

def slugify_id(name: str) -> str:
    s = name.lower().strip()
    s = re.sub(r"[()]", "", s)
    s = s.replace("ç", "c").replace("ã", "a").replace("á", "a").replace("à", "a").replace("â", "a") \
         .replace("é", "e").replace("ê", "e").replace("í", "i").replace("ó", "o").replace("ô", "o") \
         .replace("ú", "u").replace("ñ", "n")
    s = re.sub(r"[^a-z0-9]+", "-", s).strip("-")
    return s

def make_basin_obj(title: str, body: str) -> dict:
    """
    Extrai campos no estilo do seu JSON:
    climatologia, observados, anomalia, classification, prognostico, text(html)
    """
    # Extrai faixas e valores mais comuns no texto:
    # "variando entre 36 e 48 mm"
    clim = re.search(r"variando\s+entre\s+([0-9]+)\s+e\s+([0-9]+)\s*mm", body, flags=re.I)
    # "foram observados 32 mm"
    obs = re.search(r"foram\s+observados\s+([0-9]+)\s*mm", body, flags=re.I)
    # "o valor de -0.8," ou "o valor de 1.3,"
    ano = re.search(r"o\s+valor\s+de\s+(-?[0-9]+(?:\.[0-9]+)?)", body, flags=re.I)
    # "classifica a bacia em condição de X."
    cls = re.search(r"condi[cç][aã]o\s+de\s+([a-zA-ZÀ-ÿ\s]+)\.", body, flags=re.I)
    # "sugere um comportamento Y."
    prog = re.search(r"sugere\s+um\s+comportamento\s+([a-zA-ZÀ-ÿ\s]+)\.", body, flags=re.I)

    # Monta um HTML simples (parecido com seu pt.json)
    html = f"<p>{clean_text(body).replace('\n', ' ')}</p>\n"

    name = title
    # Ajustes de nomes (opcional): manter como está no PDF
    basin_id = slugify_id(name)

    out = {
        "id": basin_id,
        "name": name,
        "text": html,
        "climatologia": f"{clim.group(1)} e {clim.group(2)} mm" if clim else None,
        "observados": f"{obs.group(1)} mm" if obs else None,
        "anomalia": f"{ano.group(1)}" if ano else None,
        "classification": clean_text(cls.group(1)) if cls else None,
        "prognostico": clean_text(prog.group(1)) if prog else None,
        # "charts" você pode preencher depois, se estiver extraindo imagens
        "charts": {"acc": None, "ano": None},
    }
    return out

def parse_basin_blocks(analysis_text: str) -> list[dict]:
    """
    Divide o texto de 'Análise individual por bacia hidrográfica' em blocos:
    "Bacia do Rio X" + parágrafo(s)
    """
    # Padrões típicos do PDF:
    # "Bacia do Rio Branco"
    # "Curso principal do Rio Amazonas (Peru)"
    # "Bacias da margem esquerda do Rio Amazonas (Amazonas)"
    title_pat = re.compile(
        r"(?m)^(Bacia do Rio .+|Curso principal do Rio Amazonas \(Peru\)|Curso principal do Rio Amazonas \(Brasil\)|Curso principal do Rio Solimões|Bacia dos rios Beni e Madre de Dios|Bacia do Rio Guaporé \(Iténez\)|Bacias da margem esquerda do Rio Amazonas \(Amazonas\)|Bacias da margem esquerda do Rio Amazonas \(noroeste do Pará\)|Bacias da margem esquerda do Rio Amazonas \(nordeste do PA\))$"
    )

    titles = [(m.start(), m.group(1).strip()) for m in title_pat.finditer(analysis_text)]
    blocks = []
    for idx, (pos, title) in enumerate(titles):
        start = pos
        end = titles[idx + 1][0] if idx + 1 < len(titles) else len(analysis_text)
        block = analysis_text[start:end].strip()
        # remove o título da frente
        body = block[len(title):].strip()
        blocks.append((title, body))
    return [make_basin_obj(t, b) for t, b in blocks]

def parse_multimodel(text_pages_16_17: str) -> dict:
    # Título (primeira linha forte) e texto
    lines = [l.strip() for l in text_pages_16_17.splitlines() if l.strip()]
    title = lines[0] if lines else "Previsão multimodelo subsazonal"

    # Separar parte 7 dias e 14 dias, pegando os parágrafos que começam com "A Figura acima"
    # A depender do PDF, a parte 7 dias está na pág 13 (PDF page 16) e 14 dias na seguinte.
    parts = re.split(r"(?i)\bA Figura acima\b", text_pages_16_17)
    base_text = clean_text(parts[0].replace("\n", " ")) if parts else clean_text(text_pages_16_17)

    seven = None
    fourteen = None
    if len(parts) >= 2:
        seven = "A Figura acima" + clean_text(parts[1]).replace("\n", " ")
    if len(parts) >= 3:
        fourteen = "A Figura acima" + clean_text(parts[2]).replace("\n", " ")

    return {
        "title": title,
        "text": base_text,
        "seven_days": seven,
        "img_seven_days": None,
        "fourteen_days": fourteen,
        "img_fourteen_days": None,
    }
    
def parse_reference(text_page18: str) -> dict:
    # Página 15 do boletim (PDF page 18) contém explicação + legendas da Tabela 1
    legend_table = re.search(r"(Tabela\s*1\..+?)\n", text_page18, flags=re.I)
    legend_clim = re.search(r"(Climatologia.+?)\n", text_page18, flags=re.I)
    return {
        "text": clean_text(text_page18),
        "legend_table": legend_table.group(1).strip() if legend_table else None,
        "legend_climatology": legend_clim.group(1).strip() if legend_clim else None,
        "img_reference": None,
    }
    
def parse_anomaly_category(text_page19: str) -> dict:
    # Página 16 do boletim (PDF page 19) descreve as categorias
    m = re.search(r"Categorização.+?\n(.+)", text_page19, flags=re.I | re.S)
    desc = clean_text(m.group(1)) if m else clean_text(text_page19)
    return {
        "text": desc,
        "tables": None,
    }

In [76]:
pages = extract_pages_text(PDF_PATH)
meta = parse_front_matter(pages[0]["text"])
current_conditions = parse_current_conditions(pages[3]["text"])
# PDF pages 5-15 -> Análises por bacia
analysis_text = get_section_text(pages, start_page=5, end_page=15)
analysis = parse_basin_blocks(analysis_text)
# PDF pages 16-17 -> Multimodelo
multimodel_text = get_section_text(pages, start_page=16, end_page=17)
multimodel = parse_multimodel(multimodel_text)
# PDF page 18 -> Valores de referência
reference = parse_reference(pages[17]["text"])
# PDF page 19 -> Categorização das anomalias
anomaly_category = parse_anomaly_category(pages[18]["text"])
anomaly_behavior = [] 
out = {}
out.update({
        "doi": meta.get("doi") or template.get("doi"),
        "issn": meta.get("issn") or template.get("issn"),
        "volume": meta.get("volume") or template.get("volume"),
        "number": meta.get("number") or template.get("number"),
        "date": meta.get("date") or template.get("date"),
        "title": meta.get("title") or template.get("title"),
        "current_conditions": current_conditions,
        "analysis": analysis,
        "multimodel": multimodel,
        "reference": reference,
        "anomaly_category": anomaly_category,
        "anomaly_behavior": anomaly_behavior,
    })

In [77]:
out

{'doi': '10.61818/02910606',
 'issn': '2965-0291',
 'volume': '6',
 'number': '06',
 'date': '11 de fevereiro de 2026',
 'title': 'Boletim de monitoramento climático de grandes bacias hidrográficas: Bacia Amazônica',
 'current_conditions': {'text': 'Mapas das condições observadas de precipitação e gráficos individuais por bacias são \nproduzidos a partir dos dados MERGE/GPM gerados pelo INPE/CPTEC, considerando \ncomo climatologia para período de 2000 a 2025. Entre os dias 13 de janeiro e 11 de\nfevereiro de 2026, chuvas abaixo da climatologia caracterizaram com déficit de \nprecipitação o curso principal do Rio Amazonas em território brasileiro, bacias \nhidrográficas dos rios Abacaxis, Branco, Coari, Curuá Una, Guaporé, Iriri, Japurá, \nJuruena, Jutaí, Mamoré, bacias da margem esquerda do Rio Amazonas no nordeste e \nno noroeste do Estado do Pará, Purus, Tapajós, Teles Pires, Xingu e o curso principal \ndo Rio Solimões; chuvas acima da climatologia registradas sobre o curso principal

In [78]:
import pymupdf

In [88]:
out['analysis']

[{'id': 'bacia-do-rio-branco',
  'name': 'Bacia do Rio Branco',
  'text': '<p>A climatologia do período em análise indica chuvas com  registros variando entre 36 e 48 mm sendo considerados  normais (referência quantis 42.5% e 57.5%). Em 11 de fevereiro de 2026, foram observados 32 mm de  precipitação média acumulada sobre a bacia em 30 dias,  o cálculo da média do índice de anomalia categorizada  na área da bacia o valor de -0.8, classifica a bacia em  condição de tendência a seco. Nas próximas semanas o  comportamento climático indica elevação dos volumes de  chuva, o modelo de prognóstico subsazonal sugere um  comportamento próximo da normalidade ou tendência a chuvoso.</p>\n',
  'climatologia': '36 e 48 mm',
  'observados': '32 mm',
  'anomalia': '-0.8',
  'classification': 'tendência a seco',
  'prognostico': 'próximo da normalidade ou tendência\na chuvoso',
  'charts': {'acc': None, 'ano': None}},
 {'id': 'bacia-do-rio-negro',
  'name': 'Bacia do Rio Negro',
  'text': '<p>A climat

In [89]:
images = []
for i in out['analysis']:
    id = i['id']
    images.append(f'{id}-acc.png')
    images.append(f'{id}-ano.png')

In [92]:
len(out['analysis'])

29

In [91]:
len(images)

58

In [83]:

def get_images(path, output_path, d_boletim):
    doc = pymupdf.open(path)
    # Images Page 3
    page = doc.load_page(3)
    # mapa
    x0, y0, x1, y1 = 136.3000030517578, 414.7003479003906, 483.6500244140625, 682.5503540039062  
    rect = pymupdf.Rect(x0, y0, x1, y1)
    zoom = 3 
    mat = pymupdf.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{output_path}/map_current_conditions.png')
    # table
    x0, y0, x1, y1 = 100, 680, 515, 765   
    rect = pymupdf.Rect(x0, y0, x1, y1)
    zoom = 3 
    mat = pymupdf.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{output_path}/table_current_conditions.png')
    # Images Bacias
    images = []
    for i in d_boletim['analysis']:
        id = i['id']
        images.append(f'{id}-acc.png')
        images.append(f'{id}-ano.png')
    c = 0
    logo = (175.0500030517578, 776.2003173828125, 457.0, 832.2003173828125)
    for i in range(4, 15):
        page = doc.load_page(i)
        page_dict = page.get_text("dict") 
        blocks = page_dict.get("blocks", [])
        for b in blocks:
            btype = b.get("type", None)  
            bbox = b.get("bbox", None)
            if btype == 1 and bbox != logo:
                rect = pymupdf.Rect(bbox)
                pix = page.get_pixmap(clip=rect, dpi=200, alpha=False)
                img = images[c]
                pix.save(f'{output_path}/{img}')
                print(img)
                c += 1
    # Images Multimodel
    page = doc.load_page(15)
    x0, y0, x1, y1 = 70, 200, 515, 620   
    rect = pymupdf.Rect(x0, y0, x1, y1)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{output_path}/seven_days.png')
    page = doc.load_page(16)
    x0, y0, x1, y1 = 70, 70, 515, 500   
    rect = pymupdf.Rect(x0, y0, x1, y1)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{output_path}/fourteen_days.png')
    # Anomaly category
    doc = pymupdf.open(path)
    page = doc.load_page(18)
    x0, y0, x1, y1 = 80, 415, 530, 740   
    rect = pymupdf.Rect(x0, y0, x1, y1)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{output_path}/anomaly_table.png')
    # Anomaly Behaivor
    c = 1
    for i in range(19, 23):
        page = doc.load_page(i)
        page_dict = page.get_text("dict") 
        blocks = page_dict.get("blocks", [])
        for b in blocks:
            btype = b.get("type", None)  
            bbox = b.get("bbox", None)
            if btype == 1 and bbox != logo:
                rect = pymupdf.Rect(bbox)
                pix = page.get_pixmap(clip=rect, dpi=200, alpha=False)
                pix.save(f'bacia_{c}.png')
                c += 1



In [85]:

output_path = "/home/inacio/clima-amazonia/public/boletim/previous/6/6"
get_images(PDF_PATH, output_path, out)

bacia-do-rio-branco-acc.png
bacia-do-rio-branco-ano.png
bacia-do-rio-negro-acc.png
bacia-do-rio-negro-ano.png
bacia-do-rio-maranon-acc.png
bacia-do-rio-maranon-ano.png
bacia-do-rio-ucayali-acc.png
bacia-do-rio-ucayali-ano.png
bacia-do-rio-napo-acc.png
bacia-do-rio-napo-ano.png
curso-principal-do-rio-amazonas-peru-acc.png
curso-principal-do-rio-amazonas-peru-ano.png
bacia-do-rio-javari-acc.png
bacia-do-rio-javari-ano.png
bacia-do-rio-ica-putumayo-acc.png
bacia-do-rio-ica-putumayo-ano.png
bacia-do-rio-jutai-acc.png
bacia-do-rio-jutai-ano.png
bacia-do-rio-jurua-acc.png
bacia-do-rio-jurua-ano.png
bacia-do-rio-japura-caqueta-acc.png
bacia-do-rio-japura-caqueta-ano.png
bacia-do-rio-tefe-acc.png
bacia-do-rio-tefe-ano.png
bacia-do-rio-coari-acc.png
bacia-do-rio-coari-ano.png
bacia-do-rio-purus-acc.png
bacia-do-rio-purus-ano.png
curso-principal-do-rio-solim-es-acc.png
curso-principal-do-rio-solim-es-ano.png
bacia-dos-rios-beni-e-madre-de-dios-acc.png
bacia-dos-rios-beni-e-madre-de-dios-ano.png


IndexError: list index out of range